# Prepare image request GeoJSON

#### A subset that contains SME request parameters for image acquisaitions at sites from a vendor using the CSDA Evaluation Sites GeoJSON

Paul Montesano, PhD  
June 2026

In [3]:
import pandas as pd
import geopandas as gpd
from datetime import datetime

### Read the CSDA/Eval Sites GeoJSON stored on GitHub

+ This GeoJSON is built directly off the CSDA Evaluation Sites Database.  
+ The notebook to process this GeoJSON is here: https://github.com/pahbs/csda_summaries/blob/master/notebooks/csda_eval_sites_process.ipynb

In [4]:
TYPE = 'eval' #'csda'

In [5]:
RAW_BASE = 'https://raw.githubusercontent.com/pahbs/csda_summaries/master'
sites_url = f'{RAW_BASE}/sites/{TYPE}_sites_aoi.geojson'
sites = gpd.read_file(sites_url)

/panfs/ccds02/app/modules/jupyter/ilab/tensorflow-kernel/lib/python3.8/site-packages/pyproj/../../.././libtiff.so.6: version `LIBTIFF_4.6.1' not found (required by /app/jupyter/ilab/jupyter-lab/prod/lib/gdalplugins/../libgdal.so.36)
/panfs/ccds02/app/modules/jupyter/ilab/tensorflow-kernel/lib/python3.8/site-packages/pyproj/../../.././libtiff.so.6: version `LIBTIFF_4.6.1' not found (required by /app/jupyter/ilab/jupyter-lab/prod/lib/gdalplugins/../libgdal.so.36)
/panfs/ccds02/app/modules/jupyter/ilab/tensorflow-kernel/lib/python3.8/site-packages/pyproj/../../.././libtiff.so.6: version `LIBTIFF_4.6.1' not found (required by /app/jupyter/ilab/jupyter-lab/prod/lib/gdalplugins/../libgdal.so.36)
/panfs/ccds02/app/modules/jupyter/ilab/tensorflow-kernel/lib/python3.8/site-packages/pyproj/../../.././libtiff.so.6: version `LIBTIFF_4.6.1' not found (required by /app/jupyter/ilab/jupyter-lab/prod/lib/gdalplugins/../libgdal.so.36)
/panfs/ccds02/app/modules/jupyter/ilab/tensorflow-kernel/lib/python3

In [6]:
# Get today's date
DATE = datetime.now().strftime('%Y%m%d')
DATE

'20260923'

In [7]:
#sites.info()

### Check some useful site attributes

In [8]:
print(list(sites['Evaluation Category'].unique()))

['Geometric', 'Radiometric', 'Radiometric & Geometric', 'InSAR', 'Geometric (vert/horiz)', 'All']


### Each site's 'Site Name' is the key identifier for indicating the location of a CSDA request of data from a vendor

In [9]:
#print(list(sites['Site Name'].unique()))

#### Function to update site attributes

In [69]:
def get_site_acq_params_from_sheets(sheet_url, gid=None):
    # Convert standard edit URL to direct CSV export URL
    csv_url = re.sub(r'/edit.*', '/export?format=csv', sheet_url)
    
    # ── Append tab gid if specified ───────────────────────────────────────────
    if gid is not None:
        csv_url += f'&gid={gid}'
    
    df = pd.read_csv(csv_url)
    
    # 1. Define flexible keyword rules for each output parameter
    # The dictionary maps internal output keys to a list of potential matching keywords
    column_rules = {
        "site_name": ["sitename", "site"],
        "location_name": ["locationname", "location"],
        "country": ["country", "nation"],
        "longitude": ["longitude", "long", "lon"],
        "latitude": ["latitude", "lat"],
        "remote_sensing_domain": ["remotesensing", "domain", "sensortype"],
        "evaluation_category": ["evaluation", "category", "eval"],
        "assessment_types": ["assessment", "type"],
        "aoi_shape": ["aoishape", "shape", "geometry"],
        "max_aoi_cloud_pct": ["maxaoicloud", "aoicloud"],
        "max_scene_cloud_pct": ["maxscenecloud", "scenecloud"],
        "aoi_size_km": ["aoisize", "size"],
        "max_view_angle": ["viewangle", "angle", "pointingangle"],
        "min_num_acqs": ["minnumacqs", "minacq", "minimum"],
        "ideal_num_acqs": ["idealnumacqs", "idealacq", "ideal"]
    }
    
    # 2. Dynamic column mapping generation
    # Normalize sheet headers: "max AOI cloud %" becomes "maxaoicloud"
    normalized_headers = {
        re.sub(r'[^a-z0-9]', '', str(col).lower()): col for col in df.columns
    }
    
    resolved_mapping = {}
    for param_key, keywords in column_rules.items():
        for norm_header, raw_header in normalized_headers.items():
            # If any of our fallback keywords are found inside the normalized header, map it
            if any(kw in norm_header for kw in keywords):
                resolved_mapping[param_key] = raw_header
                break  # Stop searching for this parameter once matched

    # Fallback to make sure critical fields exist or won't crash the script
    site_col = resolved_mapping.get("site_name")
    if not site_col:
        raise ValueError(f"Could not find a column representing 'Site Name'. Headers available: {list(df.columns)}")

    site_acq_params = []
    
    # 3. Process Rows using the dynamic map
    for _, row in df.iterrows():
        if pd.isna(row[site_col]):
            continue
            
        site_name = str(row[site_col]).strip()
        
        def clean_numeric(val, is_float=False):
            if pd.isna(val): return 0
            cleaned = re.sub(r'[^\d\.\-]', '', str(val))
            if not cleaned: return 0
            return float(cleaned) if is_float else int(float(cleaned))

        def safe_get(key, default_val="", is_numeric=False, is_float=False):
            actual_col = resolved_mapping.get(key)
            if actual_col is None or actual_col not in row:
                return 0 if is_numeric else default_val
            return clean_numeric(row[actual_col], is_float) if is_numeric else str(row[actual_col]).strip()

        # Safely extract values using the flexible column dictionary
        order_params = {
            "location_name": safe_get("location_name"),
            "country": safe_get("country"),
            "longitude": safe_get("longitude", is_numeric=True, is_float=True),
            "latitude": safe_get("latitude", is_numeric=True, is_float=True),
            "remote_sensing_domain": safe_get("remote_sensing_domain"),
            "evaluation_category": safe_get("evaluation_category"),
            "assessment_types": safe_get("assessment_types"),
            "aoi_shape": safe_get("aoi_shape"),
            "max_aoi_cloud_pct": safe_get("max_aoi_cloud_pct", is_numeric=True),
            "max_scene_cloud_pct": safe_get("max_scene_cloud_pct", is_numeric=True),
            "aoi_size_km": safe_get("aoi_size_km", is_numeric=True, is_float=True),
            "max_view_angle": safe_get("max_view_angle", is_numeric=True),
            "min_num_acqs": safe_get("min_num_acqs", is_numeric=True),
            "ideal_num_acqs": safe_get("ideal_num_acqs", is_numeric=True)
        }
        
        site_acq_params.append({
            "sites": [site_name],
            "order_parameters": order_params
        })
        
    return site_acq_params
    
def update_sites_attributes(sites_gdf, site_acq_params, subset_cols_list=None):
    """
    Update sites GeoDataFrame with attributes based on configuration.
    
    Parameters:
    -----------
    sites_gdf         : GeoDataFrame - sites geodataframe to update
    site_configs      : list of dict - each with 'sites' and 'order_parameters' keys
    subset_cols_list  : list of str, optional - columns to keep in output;
                        if None, all columns are retained (default behaviour)
        
    Returns:
    --------
    GeoDataFrame : Updated sites (copy), optionally column-subsetted
    list         : All site names from configs
    """
    sites_updated = sites_gdf.copy()
    all_sites     = []
    
    for params in site_acq_params:
        site_list  = params['sites']
        attributes = params['order_parameters']
        
        mask = sites_updated['Site Name'].isin(site_list)
        for key, value in attributes.items():
            sites_updated.loc[mask, key] = value
        
        all_sites.extend(site_list)
    
    # ── Subset columns if requested ───────────────────────────────────────────
    if subset_cols_list is not None:
        # Always retain geometry so it stays a valid GeoDataFrame
        keep = [c for c in subset_cols_list if c in sites_updated.columns]
        if sites_updated.geometry.name not in keep:
            keep = keep + [sites_updated.geometry.name]
        sites_updated = sites_updated[keep]
    
    return sites_updated, all_sites

def print_site_update_report(sites_original, sites_updated, site_acq_params):

    print("=" * 70)
    print("SITE ATTRIBUTE UPDATE REPORT")
    print("=" * 70)

    available_cols = set(sites_updated.columns) & set(sites_original.columns)
    total_sites    = 0

    for i, params in enumerate(site_acq_params, 1):
        sites_list = params['sites']
        attributes = {k: v for k, v in params['order_parameters'].items()
                      if k in available_cols}

        print(f"\nGroup {i}: {len(sites_list)} site(s)")
        print("-" * 70)

        for site in sites_list:
            total_sites += 1
            print(f"\n  Site: {site}")

            orig_row    = sites_original[sites_original['Site Name'] == site]
            updated_row = sites_updated[sites_updated['Site Name'] == site]

            # ── Always print warning if site not found ────────────────────────
            if len(orig_row) == 0:
                print(f"    ⚠️  WARNING: Site not found in original dataframe")
                continue

            # ── Only print attribute diffs if any are available ───────────────
            if not attributes:
                print(f"    ℹ️  No attributes to report (columns subsetted or not matched)")
                continue

            for attr, new_value in attributes.items():
                old_value    = orig_row[attr].iloc[0] if attr in orig_row.columns else 'N/A'
                actual_value = updated_row[attr].iloc[0] if len(updated_row) > 0 else 'ERROR'
                status       = "✓" if str(actual_value) == str(new_value) else "✗"
                print(f"    {status} {attr:20s}: {old_value} → {new_value}")

    print("\n" + "=" * 70)

    # ── Compute summary stats ─────────────────────────────────────────────────
    not_found = sum(
        1 for p in site_acq_params
        for s in p['sites']
        if len(sites_original[sites_original['Site Name'] == s]) == 0
    )
    skipped = set().union(*[p['order_parameters'].keys() for p in site_acq_params]) - available_cols

    print(f"Sites considered for update:          {total_sites}")
    print(f"Sites found in dataframe:             {total_sites - not_found}")
    print(f"Sites NOT found (⚠️  warnings):        {not_found}")
    if skipped:
        print(f"CSDA acq params not requested for update:       {sorted(skipped)}")
    print("=" * 70)

### Update config of request parameters for sites chosen for this vendor request

Here is where we config & specify our 'timeseries' sites and any other types of sites we need to config & specify for this request

In [70]:
import pandas as pd
import re



### Indicate a name for the vendor

In [71]:
VENDOR_NAME = 'Tanager' # Change this
VENDOR_NAME = 'satellogic_v2' # Change this
VENDOR_NAME = 'HydroSat'
VENDOR_NAME = 'Airbus'
VENDOR_NAME = 'Vantor_Legion'

In [72]:
# --- Execution ---
SHEET_URL = "https://docs.google.com/spreadsheets/d/1lkR7cnsqq1EDISoca8tpwcdtfPvWNerbJ5EzCgq-9UI/edit?usp=sharing"
GID=0

# Misc all requests
SHEET_URL = 'https://docs.google.com/spreadsheets/d/16VopHRqlt5qbl4HgZdkVLUIHLVi9lZWPLJMY-QWsJ8g/edit?usp=sharing'

# -------- Tabs
GID=803290261 # Airbus
GID=226856298 # Legion
GID=0 # HydroSat (Kerry)
GID=918595749 # HydroSat (OR2)




In [73]:
# Central config: one entry per (vendor, sheet-tab) combination
VENDOR_SHEETS = {
    'Airbus':          ('16VopHRqlt5qbl4HgZdkVLUIHLVi9lZWPLJMY-QWsJ8g', 803290261),
    'Vantor_Legion':   ('16VopHRqlt5qbl4HgZdkVLUIHLVi9lZWPLJMY-QWsJ8g', 226856298),
    'HydroSat_Kerry':  ('16VopHRqlt5qbl4HgZdkVLUIHLVi9lZWPLJMY-QWsJ8g', 0),
    'HydroSat_OR2':    ('16VopHRqlt5qbl4HgZdkVLUIHLVi9lZWPLJMY-QWsJ8g', 918595749),
    'OroraTech_OR2':   ('16VopHRqlt5qbl4HgZdkVLUIHLVi9lZWPLJMY-QWsJ8g', 1719841802),
    'Tanager':         ('1lkR7cnsqq1EDISoca8tpwcdtfPvWNerbJ5EzCgq-9UI', 0),
    'satellogic_v2':   ('1lkR7cnsqq1EDISoca8tpwcdtfPvWNerbJ5EzCgq-9UI', 0),
}

In [74]:
def get_acq_params(vendor_name):
    """Look up sheet ID + GID for a vendor and fetch its site configs."""
    sheet_id, gid = VENDOR_SHEETS[vendor_name]
    sheet_url = f'https://docs.google.com/spreadsheets/d/{sheet_id}/edit?usp=sharing'
    return get_site_acq_params_from_sheets(sheet_url, gid=gid)

In [75]:
#SITE_CONFIGS = get_site_configs_from_sheets(SHEET_URL, gid=GID)

In [84]:
# --- Usage ---
VENDOR_NAME = 'Airbus'
SITE_CONFIGS = get_acq_params(VENDOR_NAME)

In [85]:
# sites_updated, SITES_FOR_REQUEST = update_sites_attributes(sites[['Site Name','geometry']], SITE_CONFIGS)
sites_updated, SITES_FOR_REQUEST = update_sites_attributes(sites[['Site Name','geometry']], SITE_CONFIGS
                                                           , subset_cols_list = ['Site Name','geometry'] 
                                                          )
print_site_update_report(sites, sites_updated, SITE_CONFIGS)

SITE ATTRIBUTE UPDATE REPORT

Group 1: 1 site(s)
----------------------------------------------------------------------

  Site: Baotou
    ℹ️  No attributes to report (columns subsetted or not matched)

Group 2: 1 site(s)
----------------------------------------------------------------------

  Site: Gobabeb
    ℹ️  No attributes to report (columns subsetted or not matched)

Group 3: 1 site(s)
----------------------------------------------------------------------

  Site: La Crau
    ℹ️  No attributes to report (columns subsetted or not matched)

Group 4: 1 site(s)
----------------------------------------------------------------------

  Site: PICS Libya-4
    ℹ️  No attributes to report (columns subsetted or not matched)

Group 5: 1 site(s)
----------------------------------------------------------------------

  Site: Railroad Valley
    ℹ️  No attributes to report (columns subsetted or not matched)

Group 6: 1 site(s)
----------------------------------------------------------------

In [86]:
print(SITES_FOR_REQUEST)

['Baotou', 'Gobabeb', 'La Crau', 'PICS Libya-4', 'Railroad Valley', 'Valencia', 'Golmud', 'WLEF']


### Create and write subset GeoJSON for this request

In [87]:
sites_subset = sites_updated[sites_updated['Site Name'].isin(SITES_FOR_REQUEST)]
sites_subset

,Site Name,geometry
10,Baotou,"POLYGON ((109.62968 40.85161, 109.62967 40.851..."
31,Gobabeb,"POLYGON ((15.14897 -23.60017, 15.14883 -23.602..."
48,La Crau,"POLYGON ((4.90129 43.55828, 4.90103 43.55563, ..."
51,PICS Libya-4,"POLYGON ((23.39665 28.55833, 23.39110 28.53333..."
53,Railroad Valley,"POLYGON ((-115.65561 38.49661, -115.65582 38.4..."
63,Valencia,"POLYGON ((-0.29155 39.29710, -0.29182 39.29445..."
64,Golmud,"POLYGON ((94.36205 36.39732, 94.36184 36.39467..."
66,WLEF,"POLYGON ((-90.23454 45.94397, -90.23486 45.941..."


In [88]:
DROP_COLS = ['Site Name abbrev',
             'Location Name',
 'Country',
 'Program Use',
 'Longitude',
 'Latitude',
 'Remote Sensing Domain',
 'Priority Level',
 'Evaluation Category',
 'Source',
 'Surface Domain',
 'Assessment type(s)',
 'Resolution Category',
 'aoi_shape',
 'aoi_size_km',
 # 'max_view_angle',
 # 'min_num_acqs',
 # 'ideal_num_acqs',
 'max_cloud_pct_aoi',
 'max_cloud_pct_scene',
 'contact',
 'reference',
 'notes',
 'area_km2',
 'feature_type',
 'parent_site',
 ' Location',
 'cr:id',
 'cr:lat',
 'cr:lon',
 'cr:height_above_ellipsoid_m',
 'cr:orientation_deg',
 'cr:elevation_angle_d',
 'cr:size_m',
 'geometry',
 'location_name',
 'country',
 'longitude',
 'latitude'
            ]

In [89]:
#sites_subset.drop(DROP_COLS, axis=1, inplace=True)

In [90]:
sites_subset.columns.to_list()

['Site Name', 'geometry']

In [91]:
#sites_subset.head()

In [92]:
#sites_subset.explore()

In [93]:
OUTPUT_DIR = '/home/pmontesa/code/csda_summaries/sites' # Specify your output dir here

In [94]:
sites_subset.to_file(f'{OUTPUT_DIR}/csda_sites_aoi_{VENDOR_NAME}_{DATE}.geojson')